# Data Quality Findings — Silver Layer

**Project:** Music Artists & Albums Public Perception  
**Course:** Data Analysis Programming — Semester 2026-I  
**Universidad Distrital Francisco José de Caldas**

This notebook reads the **Silver Parquet files** directly and documents all data quality observations through:
- Descriptive statistics per source
- Null rate analysis per field
- Outlier analysis (IQR method)
- Text length distributions
- NLP classification findings
- Structured Data Quality Findings Report

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT_DIR   = os.path.abspath('..')          # project root (one level up from notebooks/)
SILVER_DIR = os.path.join(ROOT_DIR, 'datalake_silver')

SILVER_REDDIT_DIR   = os.path.join(SILVER_DIR, 'reddit')
SILVER_ARTISTS_DIR  = os.path.join(SILVER_DIR, 'lastfm_top_artists')
SILVER_TRACKS_DIR   = os.path.join(SILVER_DIR, 'lastfm_top_tracks')

print('Root:          ', ROOT_DIR)
print('Silver Reddit: ', SILVER_REDDIT_DIR)
print('Silver Artists:', SILVER_ARTISTS_DIR)
print('Silver Tracks: ', SILVER_TRACKS_DIR)

---
## 1. Load Silver Parquet Files

All silver Parquet files within each folder are loaded and concatenated, preserving the daily snapshots.

In [ ]:
def load_silver_parquets(folder: str, label: str) -> pd.DataFrame:
    """Reads all Parquet files in a silver folder and returns a single DataFrame."""
    files = sorted(glob.glob(os.path.join(folder, '*.parquet')))
    if not files:
        raise FileNotFoundError(f'No Parquet files found in: {folder}')
    frames = [pd.read_parquet(f) for f in files]
    df = pd.concat(frames, ignore_index=True)
    print(f'{label}: {len(files)} file(s) → {len(df):,} rows × {df.shape[1]} columns')
    return df

df_reddit  = load_silver_parquets(SILVER_REDDIT_DIR,  'Reddit  silver')
df_artists = load_silver_parquets(SILVER_ARTISTS_DIR, 'Artists silver')
df_tracks  = load_silver_parquets(SILVER_TRACKS_DIR,  'Tracks  silver')

In [ ]:
# Quick schema preview
print('=== Reddit schema ===')
display(df_reddit.dtypes.rename('dtype').to_frame())
print('\n=== Artists schema ===')
display(df_artists.dtypes.rename('dtype').to_frame())
print('\n=== Tracks schema ===')
display(df_tracks.dtypes.rename('dtype').to_frame())

---
## 2. Descriptive Statistics

### 2.1 Reddit Silver

In [ ]:
num_cols_reddit = ['score', 'word_count', 'confidence', 'extract_confidence',
                   'score_capped', 'word_count_capped']
display(df_reddit[num_cols_reddit].describe().round(4))

**Key findings — Reddit:**
- **`score`** has high variance (posts range from 0 to 607 upvotes) → justifies IQR capping applied in silver.
- **`word_count`** ranges from 1 to extreme values → capping reduces the effective range without dropping rows.
- **`confidence`** mean ≈ 0.17 → most comments are free-form opinions, not structured artist/song recommendations.
- **`extract_confidence`** ≈ 0.0 → very few comments follow the `Artist - Song` or similar pattern, reflecting the organic nature of the discussion.

### 2.2 Last.fm Silver

In [ ]:
print('=== Last.fm Top Artists ===')
display(df_artists[['playcount', 'listeners']].describe().round(0))

print('\n=== Last.fm Top Tracks ===')
display(df_tracks[['playcount', 'listeners', 'duration_sec']].describe().round(0))

**Key findings — Last.fm:**
- **`playcount` (artists)**: ranges from millions to billions → highly right-skewed distribution; expected high outlier rate.
- **`listeners` (tracks)**: range is wide but less extreme than playcount → indicates artists with loyal followers vs. one-hit peaks.
- **`duration_sec`**: most tracks between 150–300 s (2.5–5 min), consistent with commercial releases.

In [ ]:
# Unique ingestion dates per source
for label, df in [('Reddit', df_reddit), ('Artists', df_artists), ('Tracks', df_tracks)]:
    dates = pd.to_datetime(df['ingested_at']).dt.date.nunique()
    total = len(df)
    print(f'{label:<10} | {total:>6} rows | {dates} distinct ingestion date(s)')

---
## 3. Null Rate Analysis

In [ ]:
def null_rate_table(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    """Computes null rate and 'unknown' rate per column."""
    total = len(df)
    null_counts = df.isnull().sum()
    unknown_counts = df.apply(
        lambda c: (c == 'unknown').sum() if c.dtype == object else 0
    )
    result = pd.DataFrame({
        'null_count':     null_counts,
        'null_rate_%':    (null_counts   / total * 100).round(2),
        'unknown_count':  unknown_counts,
        'unknown_rate_%': (unknown_counts / total * 100).round(2),
    })
    result.index.name = 'field'
    relevant = result[(result['null_count'] + result['unknown_count']) > 0].sort_values(
        'null_rate_%', ascending=False
    )
    print(f'\n── {source_name} ({total:,} rows) ──')
    if relevant.empty:
        print('  ✅ No nulls or unknown values in any field.')
    else:
        display(relevant)
    return result

nr_reddit  = null_rate_table(df_reddit,  'Reddit')
nr_artists = null_rate_table(df_artists, 'Last.fm Artists')
nr_tracks  = null_rate_table(df_tracks,  'Last.fm Tracks')

In [ ]:
# ── Visualise null / unknown rates ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

panels = [
    (df_reddit[['artist', 'song', 'pattern_type']],  'Reddit — Optional Fields'),
    (df_artists[['mbid']],                           'Last.fm Artists — Optional Fields'),
    (df_tracks[['mbid', 'artist_mbid']],             'Last.fm Tracks — Optional Fields'),
]

for ax, (df, title) in zip(axes, panels):
    rates = (df == 'unknown').mean() * 100
    rates = rates[rates > 0]
    if rates.empty:
        ax.text(0.5, 0.5, 'No optional fields\nwith unknown values',
                ha='center', va='center', transform=ax.transAxes, fontsize=10)
    else:
        bars = ax.barh(rates.index, rates.values, color='#e07b54', edgecolor='white')
        ax.set_xlabel('% → "unknown"')
        ax.set_xlim(0, 110)
        for bar, val in zip(bars, rates.values):
            ax.text(val + 1, bar.get_y() + bar.get_height() / 2,
                    f'{val:.1f}%', va='center', fontsize=9)
    ax.set_title(title)

plt.suptitle('Rate of optional fields filled with "unknown" by source',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

**KPI Justification — Null Rate:**  
The `artist` and `song` fields in Reddit show >95% `"unknown"` because most comments are free-form opinions without the structured `Artist - Song` format. The `mbid` field in Last.fm is also frequently unknown, indicating that not all entities are registered in MusicBrainz. Both fields are **optional** in the silver schema, so they do not affect the schema compliance rate.

---
## 4. Outlier Analysis (IQR Method)

In [ ]:
def iqr_outlier_stats(series: pd.Series, label: str) -> dict:
    """Computes IQR-based outlier statistics for a numeric series."""
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_out = ((series < lower) | (series > upper)).sum()
    return {
        'field': label,
        'Q1': round(q1, 2), 'Q3': round(q3, 2), 'IQR': round(iqr, 2),
        'lower_fence': round(lower, 2), 'upper_fence': round(upper, 2),
        'n_outliers': int(n_out),
        'outlier_rate_%': round(n_out / len(series) * 100, 2),
    }

outlier_rows = [
    iqr_outlier_stats(df_reddit['score'],            'reddit.score'),
    iqr_outlier_stats(df_reddit['word_count'],       'reddit.word_count'),
    iqr_outlier_stats(df_reddit['confidence'],       'reddit.confidence'),
    iqr_outlier_stats(df_reddit['extract_confidence'],'reddit.extract_confidence'),
    iqr_outlier_stats(df_artists['playcount'],       'artists.playcount'),
    iqr_outlier_stats(df_artists['listeners'],       'artists.listeners'),
    iqr_outlier_stats(df_tracks['playcount'],        'tracks.playcount'),
    iqr_outlier_stats(df_tracks['listeners'],        'tracks.listeners'),
    iqr_outlier_stats(df_tracks['duration_sec'],     'tracks.duration_sec'),
]

df_outliers = pd.DataFrame(outlier_rows).set_index('field')
display(df_outliers.style.background_gradient(subset=['outlier_rate_%'], cmap='YlOrRd'))

In [ ]:
# ── Boxplots ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

pairs = [
    (df_reddit['score'],              'Reddit: score'),
    (df_reddit['word_count'],         'Reddit: word_count'),
    (df_reddit['confidence'],         'Reddit: confidence'),
    (df_reddit['extract_confidence'], 'Reddit: extract_confidence'),
    (df_artists['playcount'],         'Artists: playcount'),
    (df_artists['listeners'],         'Artists: listeners'),
    (df_tracks['playcount'],          'Tracks: playcount'),
    (df_tracks['duration_sec'],       'Tracks: duration_sec'),
]

for ax, (series, title) in zip(axes.flat, pairs):
    ax.boxplot(series.dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor='#5b9bd5', alpha=0.7),
               medianprops=dict(color='#c00000', linewidth=2),
               flierprops=dict(marker='o', markerfacecolor='#e07b54',
                               markersize=4, alpha=0.5))
    ax.set_title(title, fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: (f'{x/1e9:.1f}B' if x >= 1e9
                      else f'{x/1e6:.0f}M' if x >= 1e6 else f'{x:.0f}')))
    ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('Numeric field distributions — IQR outlier detection',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

**Outlier Findings:**
- **`reddit.score`**: Viral posts (score > 500) inflate the mean. IQR capping applied in silver preserves normal variation without a single viral post distorting analyses.
- **`reddit.word_count`**: Very long comments (>50 words) are outliers — typically lengthy recommendation threads. Capping normalises them without removal.
- **`artists.playcount` / `tracks.playcount`**: Extremely right-skewed distribution (e.g. BTS with 4B+ plays). Most artists have <500M plays; raw values are correct, so no capping is applied — they appear only in relative rankings.

---
## 5. Text Length Distributions

In [ ]:
df_reddit = df_reddit.copy()
df_reddit['len_raw']   = df_reddit['raw_comment'].str.len()
df_reddit['len_clean'] = df_reddit['clean_comment'].str.len()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# raw vs clean length
axes[0].hist(df_reddit['len_raw'].dropna(),   bins=40, alpha=0.6,
             color='#5b9bd5', label='raw_comment')
axes[0].hist(df_reddit['len_clean'].dropna(), bins=40, alpha=0.6,
             color='#e07b54', label='clean_comment')
axes[0].set_xlabel('Length (characters)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Reddit: raw vs clean comment length')
axes[0].legend()

# word count
median_wc = df_reddit['word_count'].median()
mean_wc   = df_reddit['word_count'].mean()
axes[1].hist(df_reddit['word_count'].dropna(), bins=30,
             color='#70ad47', edgecolor='white')
axes[1].axvline(median_wc, color='red',    linestyle='--',
                label=f'Median: {median_wc:.0f}')
axes[1].axvline(mean_wc,   color='orange', linestyle='--',
                label=f'Mean: {mean_wc:.0f}')
axes[1].set_xlabel('Words')
axes[1].set_title('Reddit: word_count distribution')
axes[1].legend(fontsize=8)

# artist name length (Last.fm)
df_artists = df_artists.copy()
df_artists['len_name'] = df_artists['name'].str.len()
axes[2].hist(df_artists['len_name'].dropna(), bins=20,
             color='#9e69af', edgecolor='white')
axes[2].set_xlabel('Length (characters)')
axes[2].set_title('Last.fm Artists: name length')

plt.suptitle('Text length distributions by source', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────
def len_stats(series, label):
    s = series.dropna()
    return {'field': label, 'mean': round(s.mean(), 1),
            'median': round(s.median(), 1),
            'min': int(s.min()), 'max': int(s.max())}

text_stats = pd.DataFrame([
    len_stats(df_reddit['len_raw'],           'reddit.raw_comment'),
    len_stats(df_reddit['len_clean'],         'reddit.clean_comment'),
    len_stats(df_reddit['word_count'],        'reddit.word_count (tokens)'),
    len_stats(df_artists['name'].str.len(),   'artists.name'),
    len_stats(df_tracks['name'].str.len(),    'tracks.name'),
    len_stats(df_tracks['artist_name'].str.len(), 'tracks.artist_name'),
]).set_index('field')

display(text_stats)

**Finding:** The cleaning pipeline consistently reduces text length (`raw_comment` mean > `clean_comment` mean), confirming that HTML entities, URLs and punctuation are successfully removed. The median word count of ~8 tokens indicates most comments are concise reactions.

---
## 6. NLP Classification Findings

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
type_colors = {'recommendation': '#5b9bd5', 'opinion': '#70ad47',
               'mixed': '#ffc000', 'other': '#e07b54'}

# Comment type distribution
ct = df_reddit['comment_type'].value_counts()
bar_c = [type_colors.get(k, '#aaa') for k in ct.index]
bars  = axes[0].barh(ct.index, ct.values, color=bar_c, edgecolor='white')
for bar, (idx, val) in zip(bars, ct.items()):
    axes[0].text(val + 1, bar.get_y() + bar.get_height() / 2,
                 f'{val} ({val/len(df_reddit)*100:.1f}%)', va='center', fontsize=9)
axes[0].set_xlabel('Number of comments')
axes[0].set_title('Comment type distribution')
axes[0].set_xlim(0, ct.max() * 1.3)

# Pattern type breakdown
pt = df_reddit['pattern_type'].value_counts()
axes[1].bar(pt.index, pt.values, color='#5b9bd5', edgecolor='white')
axes[1].set_title('Detected music pattern type')
axes[1].set_ylabel('Count')

# Confidence distribution by comment type
for ct_val, color in type_colors.items():
    subset = df_reddit[df_reddit['comment_type'] == ct_val]['confidence']
    if len(subset) > 0:
        axes[2].hist(subset, bins=20, alpha=0.6, color=color, label=ct_val)
axes[2].set_xlabel('confidence score')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Confidence distribution by comment type')
axes[2].legend(fontsize=8)

plt.suptitle('NLP comment classification — Reddit', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print('Comment type breakdown:')
for t, n in ct.items():
    print(f'  {t:<15} {n:>4} ({n/len(df_reddit)*100:.1f}%)')

**NLP Classification Findings:**
- **~78% are opinions**: users of r/indieheads and r/hiphopheads write in discursive, unstructured prose.
- **0% pure recommendations**: the absence of the `Artist - Song` pattern in most `[FRESH ALBUM]` posts indicates users comment on the album as a whole, not specific songs.
- **Very low confidence mean (~0.17)**: consistent with the dominance of `opinion` and `other` types, which receive confidence = 0 when no music pattern is detected.
- **Implication**: sentiment analysis is more appropriate than entity extraction for this dataset.

---
## 7. Duplicate Rate Analysis

In [ ]:
def duplicate_analysis(df: pd.DataFrame, dedup_key: list, source: str):
    total      = len(df)
    exact_dups = df.duplicated().sum()
    key_dups   = df.duplicated(subset=dedup_key).sum()
    print(f'{source}')
    print(f'  Total rows:           {total:>6}')
    print(f'  Exact duplicates:     {exact_dups:>6}  ({exact_dups/total*100:.2f}%)')
    print(f'  Key duplicates {dedup_key}: {key_dups:>6}  ({key_dups/total*100:.2f}%)')
    print()

duplicate_analysis(df_reddit,  ['post_id', 'clean_comment'], 'Reddit')
duplicate_analysis(df_artists, ['name', 'ingested_at'],      'Last.fm Artists')
duplicate_analysis(df_tracks,  ['name', 'artist_name', 'ingested_at'], 'Last.fm Tracks')

---
## 8. Schema Compliance Rate

In [ ]:
# Required fields (non-nullable) per silver schema
REQUIRED = {
    'reddit': ['post_id', 'title', 'score', 'raw_comment_id', 'raw_comment',
               'clean_comment', 'tokens', 'comment_type', 'confidence',
               'has_music_pattern', 'has_contrast', 'word_count',
               'word_count_capped', 'score_capped', 'extract_confidence', 'ingested_at'],
    'artists': ['name', 'name_tokens', 'playcount', 'listeners', 'ingested_at'],
    'tracks':  ['name', 'name_tokens', 'duration_sec', 'playcount', 'listeners',
                'artist_name', 'artist_name_tokens', 'ingested_at'],
}

for source, (df, req) in zip(
    ['reddit', 'artists', 'tracks'],
    [(df_reddit, REQUIRED['reddit']),
     (df_artists, REQUIRED['artists']),
     (df_tracks, REQUIRED['tracks'])]
):
    compliant = df[req].notnull().all(axis=1).sum()
    total = len(df)
    print(f'{source:<10} schema compliance: {compliant}/{total} = {compliant/total*100:.2f}%')

---
## 9. Data Quality Findings Report

Structured summary for the Workshop 3 report — one row per issue identified.

In [ ]:
findings = [
    {
        'Source': 'Reddit', 'Field': 'artist / song',
        'Issue': 'High "unknown" rate (>95%)',
        'Frequency/Severity': '>95% of rows — Medium',
        'Handling Strategy': 'Optional fields in schema; regex extraction pipeline '
                             '(dash/by/colon) covers structured cases'
    },
    {
        'Source': 'Reddit', 'Field': 'score',
        'Issue': 'Extreme outliers (viral posts)',
        'Frequency/Severity': '~15% rows outside IQR×1.5 — High',
        'Handling Strategy': 'IQR capping → score_capped; original score preserved for traceability'
    },
    {
        'Source': 'Reddit', 'Field': 'word_count',
        'Issue': 'Extremely long comments (outliers)',
        'Frequency/Severity': '~10% of rows — Medium',
        'Handling Strategy': 'IQR capping → word_count_capped; comments not truncated'
    },
    {
        'Source': 'Reddit', 'Field': 'comment_type',
        'Issue': 'No recommendations detected (0%)',
        'Frequency/Severity': '100% opinions/other — Low (expected)',
        'Handling Strategy': 'Analysis pivot to sentiment (VADER) instead of entity extraction'
    },
    {
        'Source': 'Reddit', 'Field': 'clean_comment',
        'Issue': 'Length reduction vs raw_comment',
        'Frequency/Severity': '100% of rows — N/A (expected)',
        'Handling Strategy': 'Confirms HTML, links and punctuation removed correctly'
    },
    {
        'Source': 'Last.fm', 'Field': 'playcount',
        'Issue': 'Heavily right-skewed distribution',
        'Frequency/Severity': 'Top artists with 4B+ plays — High',
        'Handling Strategy': 'No capping applied (correct values); used in relative rankings only'
    },
    {
        'Source': 'Last.fm', 'Field': 'mbid',
        'Issue': '"unknown" when not in MusicBrainz',
        'Frequency/Severity': 'Variable ~20–40% — Low',
        'Handling Strategy': 'Optional field; filled with "unknown" in silver; no impact on business KPIs'
    },
]

pd.DataFrame(findings).set_index(['Source', 'Field'])